# Identity Inconsistency Exploration

Goal: explore whether the same `MMSI` appears with multiple vessel names before writing the final anomaly rule.

This notebook is for exploration and decision-making. The production rule will come later in `src/anomaly_rules/identity_inconsistency.py`.

## Plain-English Idea

`MMSI` is the vessel ID. `VesselName` is the vessel name being broadcast.

If one `MMSI` broadcasts multiple different names, that may be messy data, a typo, a vessel rename, bad AIS setup, or possible identity manipulation.

We do not assume suspicious behavior yet. First we profile the data.

In [ ]:
from pathlib import Path
import re

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 160)

FUSED_PATH = Path("../data/processed/AIS_2024_01_15_fused.csv")
EXPECTED_ROWS = 7_284_239

FUSED_PATH

## Load Identity Columns

The fused AIS file is large, so this notebook loads only the columns needed to explore identity conflicts.

In [ ]:
columns = [
    "MMSI",
    "BaseDateTime",
    "LAT",
    "LON",
    "VesselName",
    "IMO",
    "CallSign",
    "VesselType",
    "IMO_FLAGGED",
    "nearest_port",
    "port_distance_km",
    "near_port",
]

df = pd.read_csv(FUSED_PATH, usecols=columns)
df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"])

print(f"Rows loaded: {len(df):,}")
print(f"Expected rows: {EXPECTED_ROWS:,}")
print(f"Unique MMSIs: {df['MMSI'].nunique():,}")

df.head()

## Missing Identity Fields

Before checking conflicts, see how often `VesselName`, `CallSign`, and `IMO` are missing.

In [ ]:
missing = df[["VesselName", "CallSign", "IMO"]].isna().sum().to_frame("missing_rows")
missing["missing_pct"] = missing["missing_rows"] / len(df) * 100
missing

## Raw Vessel Name Conflicts

First pass: count raw `VesselName` values per `MMSI`.

This is intentionally naive. It treats capitalization, punctuation, and spacing differences as different names.

In [ ]:
raw_name_counts = (
    df.dropna(subset=["VesselName"])
    .groupby("MMSI")["VesselName"]
    .nunique()
    .sort_values(ascending=False)
)

print(f"MMSIs with >1 raw vessel name: {(raw_name_counts > 1).sum():,}")
raw_name_counts.value_counts().sort_index().head(20)

## Normalize Vessel Names

Normalization removes formatting noise before we decide what counts as a real name conflict.

Current normalization:
- uppercase
- strip leading/trailing spaces
- remove punctuation
- collapse repeated spaces

In [ ]:
def normalize_vessel_name(name):
    if pd.isna(name):
        return pd.NA
    normalized = str(name).upper().strip()
    normalized = re.sub(r"[^A-Z0-9 ]+", " ", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()
    if normalized == "":
        return pd.NA
    return normalized

df["normalized_vessel_name"] = df["VesselName"].apply(normalize_vessel_name)

df[["VesselName", "normalized_vessel_name"]].drop_duplicates().head(25)

## Normalized Vessel Name Conflicts

This is the better candidate for the rule: same `MMSI`, more than one normalized vessel name.

In [ ]:
normalized_name_counts = (
    df.dropna(subset=["normalized_vessel_name"])
    .groupby("MMSI")["normalized_vessel_name"]
    .nunique()
    .sort_values(ascending=False)
)

print(f"MMSIs with >1 normalized vessel name: {(normalized_name_counts > 1).sum():,}")
normalized_name_counts.value_counts().sort_index().head(20)

## Top Conflict Examples

Now inspect the highest-conflict MMSIs. This is where we decide whether conflicts are real or just formatting noise.

In [ ]:
conflicting_mmsi = normalized_name_counts[normalized_name_counts > 1].index

examples = (
    df[df["MMSI"].isin(conflicting_mmsi)]
    .groupby("MMSI")
    .agg(
        distinct_normalized_names=("normalized_vessel_name", "nunique"),
        first_seen=("BaseDateTime", "min"),
        last_seen=("BaseDateTime", "max"),
        row_count=("MMSI", "size"),
        distinct_callsigns=("CallSign", "nunique"),
        distinct_imos=("IMO", "nunique"),
        imo_flagged_rows=("IMO_FLAGGED", "sum"),
    )
    .sort_values(["distinct_normalized_names", "row_count"], ascending=False)
)

examples.head(25)

## Inspect One MMSI Deeply

Change `mmsi_to_inspect` after looking at the table above.

In [ ]:
mmsi_to_inspect = examples.index[0]

name_breakdown = (
    df[df["MMSI"] == mmsi_to_inspect]
    .groupby(["VesselName", "normalized_vessel_name"], dropna=False)
    .agg(
        rows=("MMSI", "size"),
        first_seen=("BaseDateTime", "min"),
        last_seen=("BaseDateTime", "max"),
        callsigns=("CallSign", lambda s: sorted(set(s.dropna().astype(str)))),
        imos=("IMO", lambda s: sorted(set(s.dropna().astype(str)))),
        sample_lat=("LAT", "median"),
        sample_lon=("LON", "median"),
    )
    .sort_values("rows", ascending=False)
)

mmsi_to_inspect, name_breakdown

## Decision Notes

Use this section for your own notes.

Questions:
- How many raw conflicts disappear after normalization?
- Do the remaining conflicts look like real different vessel names?
- Do name conflicts also have CallSign or IMO conflicts?
- What should count as low, medium, or high suspicion?

In [2]:
"""
profile_identity.py

Explore AIS identity inconsistency before writing the final anomaly rule.

Goal:
Find cases where the same MMSI broadcasts more than one vessel name.

This script does NOT create final anomaly events yet.
It only prints counts and examples so we can decide the rule carefully.
"""

from pathlib import Path
import re

import pandas as pd


FUSED_PATH = Path(__file__).parent.parent.parent / "data" / "processed" / "AIS_2024_01_15_fused.csv"
EXPECTED_ROWS = 7_284_239


def normalize_vessel_name(name):
    """Normalize vessel names so formatting differences do not create fake conflicts."""
    if pd.isna(name):
        return pd.NA

    normalized = str(name).upper().strip()
    normalized = re.sub(r"[^A-Z0-9 ]+", " ", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()

    if normalized == "":
        return pd.NA

    return normalized


def unique_values(series):
    """Return sorted non-null unique values as a readable list."""
    values = sorted(set(series.dropna().astype(str)))
    return values


def main():
    columns = [
        "MMSI",
        "BaseDateTime",
        "LAT",
        "LON",
        "VesselName",
        "IMO",
        "IMO_FLAGGED",
        "CallSign",
        "VesselType",
        "nearest_port",
        "port_distance_km",
        "near_port",
    ]

    print(f"Loading fused AIS data from {FUSED_PATH}...")
    df = pd.read_csv(FUSED_PATH, usecols=columns)
    df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"])

    print("\n--- BASIC COUNTS ---")
    print(f"Rows loaded:        {len(df):,}")
    print(f"Expected rows:      {EXPECTED_ROWS:,}")
    print(f"Unique MMSIs:       {df['MMSI'].nunique():,}")
    print(f"Missing VesselName: {df['VesselName'].isna().sum():,}")
    print(f"Missing CallSign:   {df['CallSign'].isna().sum():,}")
    print(f"Missing IMO:        {df['IMO'].isna().sum():,}")

    if len(df) != EXPECTED_ROWS:
        print("WARNING: unexpected row count. Check that the fused file was used.")

    print("\n--- RAW NAME CONFLICTS ---")
    raw_name_counts = (
        df.dropna(subset=["VesselName"])
        .groupby("MMSI")["VesselName"]
        .nunique()
        .sort_values(ascending=False)
    )

    raw_conflict_count = int((raw_name_counts > 1).sum())
    print(f"MMSIs with >1 raw VesselName: {raw_conflict_count:,}")
    print("\nRaw distinct-name count distribution:")
    print(raw_name_counts.value_counts().sort_index().head(20).to_string())

    print("\n--- NORMALIZED NAME CONFLICTS ---")
    df["normalized_vessel_name"] = df["VesselName"].apply(normalize_vessel_name)

    normalized_name_counts = (
        df.dropna(subset=["normalized_vessel_name"])
        .groupby("MMSI")["normalized_vessel_name"]
        .nunique()
        .sort_values(ascending=False)
    )

    normalized_conflict_count = int((normalized_name_counts > 1).sum())
    print(f"MMSIs with >1 normalized VesselName: {normalized_conflict_count:,}")
    print("\nNormalized distinct-name count distribution:")
    print(normalized_name_counts.value_counts().sort_index().head(20).to_string())

    conflicts_removed = raw_conflict_count - normalized_conflict_count
    print(f"\nConflicts removed by normalization: {conflicts_removed:,}")

    print("\n--- TOP CONFLICTING MMSIS ---")
    conflicting_mmsi = normalized_name_counts[normalized_name_counts > 1].index

    examples = (
        df[df["MMSI"].isin(conflicting_mmsi)]
        .groupby("MMSI")
        .agg(
            distinct_normalized_names=("normalized_vessel_name", "nunique"),
            row_count=("MMSI", "size"),
            first_seen=("BaseDateTime", "min"),
            last_seen=("BaseDateTime", "max"),
            distinct_callsigns=("CallSign", "nunique"),
            distinct_imos=("IMO", "nunique"),
            imo_flagged_rows=("IMO_FLAGGED", "sum"),
            sample_lat=("LAT", "median"),
            sample_lon=("LON", "median"),
        )
        .sort_values(["distinct_normalized_names", "row_count"], ascending=False)
    )

    print(examples.head(20).to_string())

    print("\n--- DETAILED EXAMPLES ---")
    for mmsi in examples.head(5).index:
        print(f"\nMMSI: {mmsi}")

        breakdown = (
            df[df["MMSI"] == mmsi]
            .groupby(["VesselName", "normalized_vessel_name"], dropna=False)
            .agg(
                rows=("MMSI", "size"),
                first_seen=("BaseDateTime", "min"),
                last_seen=("BaseDateTime", "max"),
                callsigns=("CallSign", unique_values),
                imos=("IMO", unique_values),
                median_lat=("LAT", "median"),
                median_lon=("LON", "median"),
            )
            .sort_values("rows", ascending=False)
        )

        print(breakdown.to_string())

    print("\n--- DECISION QUESTIONS ---")
    print("1. Did normalization remove many conflicts?")
    print("2. Do the remaining conflicts look like real different names?")
    print("3. Do conflicting names also have conflicting CallSigns or IMOs?")
    print("4. Should the final rule use low/medium/high suspicion levels?")


if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined

In [3]:
from pathlib import Path
import re

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 160)

# Notebook-safe path.
# If your notebook is running from the project root, use Path.cwd().
# If it is running from notebooks/, use Path.cwd().parent.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FUSED_PATH = PROJECT_ROOT / "data" / "processed" / "AIS_2024_01_15_fused.csv"
EXPECTED_ROWS = 7_284_239

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Fused path:", FUSED_PATH)
print("Fused file exists:", FUSED_PATH.exists())

Current working directory: C:\Users\Adrian\Projects\maritime-anomaly-pipeline\notebooks
Project root: C:\Users\Adrian\Projects\maritime-anomaly-pipeline
Fused path: C:\Users\Adrian\Projects\maritime-anomaly-pipeline\data\processed\AIS_2024_01_15_fused.csv
Fused file exists: True


In [4]:
columns = [
    "MMSI",
    "BaseDateTime",
    "LAT",
    "LON",
    "VesselName",
    "IMO",
    "IMO_FLAGGED",
    "CallSign",
    "VesselType",
    "nearest_port",
    "port_distance_km",
    "near_port",
]

df = pd.read_csv(FUSED_PATH, usecols=columns)
df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"])

print("\n--- BASIC COUNTS ---")
print(f"Rows loaded:        {len(df):,}")
print(f"Expected rows:      {EXPECTED_ROWS:,}")
print(f"Unique MMSIs:       {df['MMSI'].nunique():,}")
print(f"Missing VesselName: {df['VesselName'].isna().sum():,}")
print(f"Missing CallSign:   {df['CallSign'].isna().sum():,}")
print(f"Missing IMO:        {df['IMO'].isna().sum():,}")

df.head()


--- BASIC COUNTS ---
Rows loaded:        7,284,239
Expected rows:      7,284,239
Unique MMSIs:       15,135
Missing VesselName: 9,355
Missing CallSign:   792,048
Missing IMO:        2,253,648


,MMSI,BaseDateTime,LAT,LON,VesselName,IMO,CallSign,VesselType,IMO_FLAGGED,nearest_port,port_distance_km,near_port
0,338467439,2024-01-15 00:00:00,48.75657,-122.50736,NEVE,IMO0000000,NaN,36.0,True,Bellingham,0.908207,True
1,367090390,2024-01-15 00:00:00,29.06215,-90.16063,SUN DAY,NaN,WDC8625,60.0,False,Loop Terminal,24.320419,True
2,366709780,2024-01-15 00:00:00,47.79480,-122.49430,WSF SPOKANE,IMO7214325,WYX2004,60.0,False,Edwards Point,7.067493,True
3,366943960,2024-01-15 00:00:00,33.35171,-118.26675,CATALINA EXPRESS,IMO8967888,WCJ6210,40.0,False,Avalon,4.640354,True
4,338391523,2024-01-15 00:00:01,48.49284,-122.68201,RESOLUTE,IMO0000000,NaN,37.0,True,Anacortes,5.494936,True


In [5]:
def normalize_vessel_name(name):
    """Normalize vessel names so formatting differences do not create fake conflicts."""
    if pd.isna(name):
        return pd.NA

    normalized = str(name).upper().strip()
    normalized = re.sub(r"[^A-Z0-9 ]+", " ", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()

    if normalized == "":
        return pd.NA

    return normalized


def unique_values(series):
    """Return sorted non-null unique values as a readable list."""
    return sorted(set(series.dropna().astype(str)))


df["normalized_vessel_name"] = df["VesselName"].apply(normalize_vessel_name)

In [6]:
print("\n--- RAW NAME CONFLICTS ---")

raw_name_counts = (
    df.dropna(subset=["VesselName"])
    .groupby("MMSI")["VesselName"]
    .nunique()
    .sort_values(ascending=False)
)

raw_conflict_count = int((raw_name_counts > 1).sum())

print(f"MMSIs with >1 raw VesselName: {raw_conflict_count:,}")
print("\nRaw distinct-name count distribution:")
print(raw_name_counts.value_counts().sort_index().head(20).to_string())


--- RAW NAME CONFLICTS ---
MMSIs with >1 raw VesselName: 0

Raw distinct-name count distribution:
VesselName
1    15034


In [7]:
print("\n--- NORMALIZED NAME CONFLICTS ---")

normalized_name_counts = (
    df.dropna(subset=["normalized_vessel_name"])
    .groupby("MMSI")["normalized_vessel_name"]
    .nunique()
    .sort_values(ascending=False)
)

normalized_conflict_count = int((normalized_name_counts > 1).sum())

print(f"MMSIs with >1 normalized VesselName: {normalized_conflict_count:,}")
print("\nNormalized distinct-name count distribution:")
print(normalized_name_counts.value_counts().sort_index().head(20).to_string())

conflicts_removed = raw_conflict_count - normalized_conflict_count
print(f"\nConflicts removed by normalization: {conflicts_removed:,}")


--- NORMALIZED NAME CONFLICTS ---
MMSIs with >1 normalized VesselName: 0

Normalized distinct-name count distribution:
normalized_vessel_name
1    15034

Conflicts removed by normalization: 0


In [8]:
print("\n--- TOP CONFLICTING MMSIS ---")

conflicting_mmsi = normalized_name_counts[normalized_name_counts > 1].index

examples = (
    df[df["MMSI"].isin(conflicting_mmsi)]
    .groupby("MMSI")
    .agg(
        distinct_normalized_names=("normalized_vessel_name", "nunique"),
        row_count=("MMSI", "size"),
        first_seen=("BaseDateTime", "min"),
        last_seen=("BaseDateTime", "max"),
        distinct_callsigns=("CallSign", "nunique"),
        distinct_imos=("IMO", "nunique"),
        imo_flagged_rows=("IMO_FLAGGED", "sum"),
        sample_lat=("LAT", "median"),
        sample_lon=("LON", "median"),
    )
    .sort_values(["distinct_normalized_names", "row_count"], ascending=False)
)

examples.head(20)


--- TOP CONFLICTING MMSIS ---


,distinct_normalized_names,row_count,first_seen,last_seen,distinct_callsigns,distinct_imos,imo_flagged_rows,sample_lat,sample_lon
MMSI,,,,,,,,,


In [9]:
print("\n--- SUPPORTING IDENTITY FIELD CONFLICTS ---")

for col in ["VesselName", "CallSign", "IMO"]:
    counts = df.dropna(subset=[col]).groupby("MMSI")[col].nunique()
    conflict_count = int((counts > 1).sum())

    print(f"{col}: MMSIs with >1 non-null value = {conflict_count:,}")
    print(counts.value_counts().sort_index().head(10).to_string())
    print()


--- SUPPORTING IDENTITY FIELD CONFLICTS ---
VesselName: MMSIs with >1 non-null value = 0
VesselName
1    15034

CallSign: MMSIs with >1 non-null value = 0
CallSign
1    12170

IMO: MMSIs with >1 non-null value = 0
IMO
1    12069

